In [ ]:
import os, sys, warnings
from pathlib import Path
_r = Path.cwd()
while not (_r / 'requirements.txt').exists() and _r != _r.parent:
    _r = _r.parent
os.chdir(_r); sys.path.insert(0, str(_r / 'src')); warnings.filterwarnings('ignore')

# Feature analysis — pruning + SHAP (XGBoost monthly-loss model)

Features after pruning (`src/model.py`): dropped `volt_min` (≈constant), `temp_mean` & `frac_drive`
(sparse/noisy for Mahindra) — backtest improved. Below: importance + **SHAP** for per-prediction
explanations of monthly SoH loss.

In [ ]:
import numpy as np, pandas as pd, xgboost as xgb, shap
import matplotlib.pyplot as plt
import model

m = pd.read_parquet('data/mahindra/features/feature_table.parquet').sort_values(['vin', 'month'])
tr = model.build_transitions(m)
X = tr[model.FEATS]; y = tr['loss'].to_numpy(); w = tr['w'].to_numpy()
xm = xgb.XGBRegressor(n_estimators=300, learning_rate=0.03, max_depth=4, subsample=0.8,
                      colsample_bytree=0.8, n_jobs=8, verbosity=0).fit(X.to_numpy(), y, sample_weight=w)
print(f'{len(model.FEATS)} features, {len(tr)} training transitions (target = monthly SoH loss pp/mo)')
print('features:', model.FEATS)

## SHAP — global feature impact (beeswarm + mean |SHAP|)

In [ ]:
explainer = shap.TreeExplainer(xm)
sv = explainer(X)
sv.feature_names = model.FEATS
shap.plots.beeswarm(sv, max_display=len(model.FEATS), show=False)
plt.title('SHAP — impact of each feature on predicted monthly SoH loss'); plt.tight_layout(); plt.show()

In [ ]:
shap.plots.bar(sv, max_display=len(model.FEATS), show=False)
plt.title('Mean |SHAP| — average impact magnitude'); plt.tight_layout(); plt.show()

## Per-prediction explanation (waterfall)

Why the model predicts the loss it does for a single high-degradation month — each feature's
contribution above/below the base rate.

In [ ]:
# pick a high-predicted-loss row to explain
j = int(np.argmax(xm.predict(X.to_numpy())))
print(f'explaining row with predicted loss {xm.predict(X.to_numpy())[j]:.2f} pp/month')
shap.plots.waterfall(sv[j], max_display=12, show=False)
plt.tight_layout(); plt.show()